# Active Dwarf Galaxy Database: grahspj SED Fits and $\alpha_{\mathrm{OX}}$

This notebook downloads the Active Dwarf Galaxy Database catalog from GitHub, keeps sources with detected X-ray luminosities (`upper_limit_flag == False`), fits each usable AGN SED with `grahspj`, and compares the inferred UV-to-X-ray slope against the $L_{2500}$ relation.

The UV luminosity is inferred from the fitted rest-frame AGN component at 2500 A. The X-ray monochromatic luminosity at 2 keV is inferred from the catalog 0.5-8 keV luminosity assuming a power-law photon index.

## Setup

This notebook assumes:
- you are running from the `notebooks/` directory of this repository
- the `sed` conda environment or equivalent dependencies are available
- a valid DSPS SSP file is available as a sibling checkout at `../../jaxqsofit/tempdata.h5`

The default fitting loop uses MAP optimization (`fit_method="optax"`) so the full catalog run is tractable. Increase `OPTAX_STEPS` or switch to `optax+nuts` for a smaller subset if you need posterior uncertainties.

In [ ]:
from pathlib import Path
import sys
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from tqdm.auto import tqdm

project_root = Path.cwd().resolve().parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from grahspj.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from grahspj.core import GRAHSPJ
from grahspj.mplstyle import style_path

plt.style.use(style_path())


In [ ]:
CATALOG_URL = "https://raw.githubusercontent.com/erikwasleske/Active-Dwarf-Galaxy-Database/main/ADGD_v1.fits"
data_dir = project_root / "data" / "adgd"
data_dir.mkdir(parents=True, exist_ok=True)
catalog_path = data_dir / "ADGD_v1.fits"

if not catalog_path.is_file():
    print("Downloading", CATALOG_URL)
    urlretrieve(CATALOG_URL, catalog_path)

catalog = Table.read(catalog_path)
print(f"Catalog rows: {len(catalog):,}")
catalog[:3]


In [ ]:
def finite_positive(values):
    arr = np.asarray(values, dtype=float)
    return np.isfinite(arr) & (arr > 0.0)

detected_xray = (
    (~np.asarray(catalog["upper_limit_flag"], dtype=bool))
    & finite_positive(catalog["L_0.5-8"])
    & finite_positive(catalog["Redshift"])
)
adgd = catalog[detected_xray]

print(f"X-ray detections with upper_limit_flag == False: {len(adgd):,}")
adgd[["ID", "RA", "DEC", "Redshift", "Mass", "L_0.5-8", "upper_limit_flag"]][:10]


## Catalog Photometry

The catalog already includes homogeneous survey photometry used for SED modeling in the ADGD work. This notebook uses the filters already supported by the current `grahspj` examples: GALEX FUV/NUV, SDSS ugriz, and WISE W1-W4.

SDSS and GALEX magnitudes are treated as AB. WISE magnitudes are converted from Vega using the standard AllWISE zero-points in Jy.

In [ ]:
WISE_ZEROPOINT_JY = {
    "W1": 309.540,
    "W2": 171.787,
    "W3": 31.674,
    "W4": 8.363,
}

PHOTOMETRY_COLUMNS = [
    {"filter_name": "FUV_galex", "speclite_name": "galex-fuv", "mag": "FUVmag", "err": "e_FUVmag", "system": "ab", "psf_fwhm_arcsec": 4.3},
    {"filter_name": "NUV_galex", "speclite_name": "galex-nuv", "mag": "NUVmag", "err": "e_NUVmag", "system": "ab", "psf_fwhm_arcsec": 5.3},
    {"filter_name": "u_sdss", "speclite_name": "sdss2010-u", "mag": "cModelMag_u", "err": "cModelMagErr_u", "ext": "extinction_u", "system": "ab", "psf_fwhm_arcsec": 1.53},
    {"filter_name": "g_sdss", "speclite_name": "sdss2010-g", "mag": "cModelMag_g", "err": "cModelMagErr_g", "ext": "extinction_g", "system": "ab", "psf_fwhm_arcsec": 1.44},
    {"filter_name": "r_sdss", "speclite_name": "sdss2010-r", "mag": "cModelMag_r", "err": "cModelMagErr_r", "ext": "extinction_r", "system": "ab", "psf_fwhm_arcsec": 1.32},
    {"filter_name": "i_sdss", "speclite_name": "sdss2010-i", "mag": "cModelMag_i", "err": "cModelMagErr_i", "ext": "extinction_i", "system": "ab", "psf_fwhm_arcsec": 1.26},
    {"filter_name": "z_sdss", "speclite_name": "sdss2010-z", "mag": "cModelMag_z", "err": "cModelMagErr_z", "ext": "extinction_z", "system": "ab", "psf_fwhm_arcsec": 1.29},
    {"filter_name": "W1", "speclite_name": "wise2010-W1", "mag": "w1mpro_new", "err": "w1sigmpro_new", "system": "vega", "psf_fwhm_arcsec": 6.08},
    {"filter_name": "W2", "speclite_name": "wise2010-W2", "mag": "w2mpro_new", "err": "w2sigmpro_new", "system": "vega", "psf_fwhm_arcsec": 6.84},
    {"filter_name": "W3", "speclite_name": "wise2010-W3", "mag": "w3mpro_new", "err": "w3sigmpro_new", "system": "vega", "psf_fwhm_arcsec": 7.36},
    {"filter_name": "W4", "speclite_name": "wise2010-W4", "mag": "w4mpro_new", "err": "w4sigmpro_new", "system": "vega", "psf_fwhm_arcsec": 11.99},
]

def as_float(value):
    try:
        out = float(value)
    except Exception:
        return np.nan
    return out if np.isfinite(out) else np.nan

def mag_to_flux_mjy(mag, system, filter_name):
    if system == "ab":
        return 3631.0e3 * 10.0 ** (-0.4 * mag)
    if system == "vega":
        return WISE_ZEROPOINT_JY[filter_name] * 1.0e3 * 10.0 ** (-0.4 * mag)
    raise ValueError(f"Unknown magnitude system: {system}")

def magerr_to_fluxerr_mjy(flux_mjy, mag_err):
    return abs(flux_mjy) * np.log(10.0) * 0.4 * mag_err

def photometry_for_row(row, min_frac_err=0.03):
    rows = []
    for spec in PHOTOMETRY_COLUMNS:
        mag = as_float(row[spec["mag"]])
        mag_err = as_float(row[spec["err"]])
        if not np.isfinite(mag) or not np.isfinite(mag_err) or mag_err <= 0.0:
            continue
        if "ext" in spec:
            mag = mag - max(as_float(row[spec["ext"]]), 0.0)
        flux_mjy = mag_to_flux_mjy(mag, spec["system"], spec["filter_name"])
        err_mjy = max(magerr_to_fluxerr_mjy(flux_mjy, mag_err), min_frac_err * abs(flux_mjy))
        if not np.isfinite(flux_mjy) or not np.isfinite(err_mjy) or flux_mjy <= 0.0 or err_mjy <= 0.0:
            continue
        rows.append({
            "filter_name": spec["filter_name"],
            "speclite_name": spec["speclite_name"],
            "flux_mjy": flux_mjy,
            "err_mjy": err_mjy,
            "psf_fwhm_arcsec": spec["psf_fwhm_arcsec"],
        })
    return rows

phot_counts = np.array([len(photometry_for_row(row)) for row in adgd])
plt.figure(figsize=(7, 4))
plt.hist(phot_counts, bins=np.arange(phot_counts.max() + 2) - 0.5)
plt.xlabel("usable grahspj photometric bands")
plt.ylabel("number of X-ray detections")
plt.show()

MIN_BANDS_TO_FIT = 5
fit_rows = adgd[phot_counts >= MIN_BANDS_TO_FIT]
print(f"Sources with >= {MIN_BANDS_TO_FIT} usable bands: {len(fit_rows):,}")


## Fit Configuration

`MAX_SOURCES = None` runs all usable X-ray detections. Set it to a small integer while developing the notebook.

In [ ]:
dsps_ssp_fn = project_root.parent / "jaxqsofit" / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

MAX_SOURCES = None
FIT_METHOD = "optax"
OPTAX_STEPS = 500
OPTAX_LR = 5e-3

def source_id(row):
    return str(row["ID"]).strip().replace(" ", "_").replace("/", "_")

def build_fit_config(row):
    phot_rows = photometry_for_row(row)
    mass = as_float(row["Mass"])
    mass_loc = mass if np.isfinite(mass) else 9.0
    return FitConfig(
        observation=Observation(
            object_id=source_id(row),
            redshift=float(row["Redshift"]),
            fit_redshift=False,
            ra=as_float(row["RA"]),
            dec=as_float(row["DEC"]),
        ),
        photometry=PhotometryData(
            filter_names=[item["filter_name"] for item in phot_rows],
            fluxes=[item["flux_mjy"] for item in phot_rows],
            errors=[item["err_mjy"] for item in phot_rows],
            is_upper_limit=[False] * len(phot_rows),
            psf_fwhm_arcsec=[item["psf_fwhm_arcsec"] for item in phot_rows],
        ),
        filters=FilterSet(
            speclite_names={item["filter_name"]: item["speclite_name"] for item in phot_rows},
            use_grahsp_database=False,
        ),
        galaxy=GalaxyConfig(
            dsps_ssp_fn=str(dsps_ssp_fn),
            n_wave=768,
            sfh_n_steps=48,
        ),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(use_host_capture_model=True),
        inference=InferenceConfig(map_steps=OPTAX_STEPS, learning_rate=OPTAX_LR, seed=0),
        prior_config={
            "log_stellar_mass": {"loc": mass_loc, "scale": 0.6},
            "fracAGN_5100": {"loc": 0.5, "scale": 0.25},
            "ebv_gal": {"scale": 0.2},
            "ebv_agn": {"scale": 0.2},
        },
    )

rows_to_fit = fit_rows[:MAX_SOURCES] if MAX_SOURCES is not None else fit_rows
print(f"Will fit {len(rows_to_fit):,} sources")


In [ ]:
fitters = {}
fit_summaries = []
failed = []

for row in tqdm(rows_to_fit, desc="grahspj fits"):
    sid = source_id(row)
    try:
        cfg = build_fit_config(row)
        fitter = GRAHSPJ(cfg)
        fitter.fit(
            fit_method=FIT_METHOD,
            prior_config=cfg.prior_config,
            dsps_ssp_fn=cfg.galaxy.dsps_ssp_fn,
            optax_steps=cfg.inference.map_steps,
            optax_lr=cfg.inference.learning_rate,
            plot_fig=False,
            save_fig=False,
            save_result=False,
            progress_bar=False,
        )
        fitters[sid] = fitter
        fit_summaries.append({"ID": sid, "n_bands": len(cfg.photometry.filter_names), **fitter.summary()})
    except Exception as exc:
        failed.append({"ID": sid, "error": repr(exc)})

summary_table = Table(rows=fit_summaries)
failed_table = Table(rows=failed) if failed else Table(names=("ID", "error"))

print(f"Successful fits: {len(summary_table):,}")
print(f"Failed fits: {len(failed_table):,}")
summary_table[:5]


In [ ]:
failed_table[:10]


## Derive $L_{2500}$ and $\alpha_{\mathrm{OX}}$

$\alpha_{\mathrm{OX}} = 0.3838 \log_{10}(L_{\nu,2\,\mathrm{keV}} / L_{\nu,2500\,\mathring{A}})$.

The catalog provides integrated 0.5-8 keV luminosity. The conversion to 2 keV monochromatic luminosity assumes $L_E \propto E^{1-\Gamma}$ with `PHOTON_INDEX = 1.8`.

In [ ]:
C_A_PER_S = 2.99792458e18
H_KEV_S = 4.135667696e-18
PHOTON_INDEX = 1.8
UV_COMPONENT_SITE = "agn_rest_sed"

row_by_id = {source_id(row): row for row in rows_to_fit}

def median_site(pred, key):
    arr = np.asarray(pred[key], dtype=float)
    return np.nanmedian(arr, axis=0) if arr.ndim > 1 else arr

def lnu_2500_from_fit(fitter, component_site=UV_COMPONENT_SITE):
    pred = fitter.predict()
    rest_wave = median_site(pred, "rest_wave")
    component_llambda_w_a = median_site(pred, component_site)
    llambda_2500_w_a = float(np.interp(2500.0, rest_wave, component_llambda_w_a, left=np.nan, right=np.nan))
    return llambda_2500_w_a * 1.0e7 * 2500.0**2 / C_A_PER_S

def lnu_2kev_from_lx(lx_erg_s, gamma=PHOTON_INDEX, e1=0.5, e2=8.0, e0=2.0):
    lx = float(lx_erg_s)
    if np.isclose(gamma, 2.0):
        norm = lx / np.log(e2 / e1)
    else:
        norm = lx * (2.0 - gamma) / (e2 ** (2.0 - gamma) - e1 ** (2.0 - gamma))
    l_e_2kev = norm * e0 ** (1.0 - gamma)
    return l_e_2kev * H_KEV_S

alpha_rows = []
for sid, fitter in tqdm(fitters.items(), desc="alpha_ox"):
    row = row_by_id[sid]
    lnu_2500 = lnu_2500_from_fit(fitter)
    lnu_2kev = lnu_2kev_from_lx(row["L_0.5-8"])
    if not (np.isfinite(lnu_2500) and np.isfinite(lnu_2kev) and lnu_2500 > 0.0 and lnu_2kev > 0.0):
        continue
    alpha_ox = 0.3838 * np.log10(lnu_2kev / lnu_2500)
    alpha_rows.append({
        "ID": sid,
        "Redshift": float(row["Redshift"]),
        "Mass": as_float(row["Mass"]),
        "L_0.5-8": float(row["L_0.5-8"]),
        "log_Lnu_2500": np.log10(lnu_2500),
        "log_Lnu_2kev": np.log10(lnu_2kev),
        "alpha_ox": alpha_ox,
    })

alpha_table = Table(rows=alpha_rows)
print(f"Rows with finite alpha_ox: {len(alpha_table):,}")
alpha_table[:10]


In [ ]:
def just_relation_alpha_ox(log_l2500):
    # Just et al. 2007, alpha_ox = (-0.137) log Lnu(2500 A) + 2.638
    return -0.137 * np.asarray(log_l2500, dtype=float) + 2.638

fig, ax = plt.subplots(figsize=(7, 5))
x = np.asarray(alpha_table["log_Lnu_2500"], dtype=float)
y = np.asarray(alpha_table["alpha_ox"], dtype=float)
sc = ax.scatter(
    x,
    y,
    c=np.asarray(alpha_table["Mass"], dtype=float),
    cmap="viridis",
    s=34,
    alpha=0.85,
    edgecolor="none",
)

if len(x):
    xx = np.linspace(np.nanmin(x) - 0.2, np.nanmax(x) + 0.2, 200)
    ax.plot(xx, just_relation_alpha_ox(xx), color="k", lw=2, label="Just+2007")

ax.set_xlabel(r"$\log_{10} L_{\nu}(2500\,\AA)$ [erg s$^{-1}$ Hz$^{-1}$]")
ax.set_ylabel(r"$\alpha_{\mathrm{OX}}$")
ax.legend(frameon=False)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label(r"$\log_{10} M_\star / M_\odot$")
plt.show()


In [ ]:
output_path = data_dir / "adgd_grahspj_alpha_ox.ecsv"
alpha_table.write(output_path, overwrite=True)
print("Wrote", output_path)


## Notes

- The plotted UV luminosity uses `UV_COMPONENT_SITE = "agn_rest_sed"`. Use `"disk_rest_sed"` if you want only the fitted disk continuum at 2500 A.
- The X-ray conversion assumes `PHOTON_INDEX = 1.8`. Changing that value shifts all inferred $L_{\nu,2\,\mathrm{keV}}$ values coherently.
- The ADGD catalog includes CIGALE-derived `bayes.agn.intrin_Lnu_2500A_30deg`; this notebook does not use it for the main plot because the requested UV luminosity is inferred from the `grahspj` AGN component.
